In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mrl_trace import paths
from mrl_trace.device import TransientGate, tau_r
from mrl_trace.stats import bootstrap_ci

# QUICK=True re-runs a fast, few-seed version of an experiment in-kernel (serial, no Pool);
# the default (QUICK=False) REPLAYS the committed 20-seed grid so every figure renders
# instantly. Heavy sweeps (exp8's dense 13x18 grid) always document a `--full` shell
# command instead and are never a slow default cell.
QUICK = False

# device retention curves are coloured by a sequential viridis (long->short), matching
# the manuscript trace-window figure; controls/fits use the shared accent palette.
GREEN, INDIGO, RED, GOLD, GREY, INK = "#3aa07a", "#2f4b8f", "#c0392b", "#e0a93b", "#9aa6b2", "#2b2b2b"
VIR = [plt.cm.viridis(x) for x in (0.85, 0.5, 0.15)]   # tau_leak long -> short

def _clean(ax):
    for sp in ("top", "right"):
        ax.spines[sp].set_visible(False)
    ax.set_axisbelow(True); ax.grid(True, color="0.88", lw=0.5)

print("data/results:", paths.results_dir())
print(f"operating point V=1.5 -> fitted rise time tau_r = {tau_r(1.5):.2f} s")

In [ ]:
# Pure device trace: deterministic, instant, computed inline (no grid replay).
t = np.arange(0.0, 80.0, 0.05)          # s
coincidence_at, coincidence_dur = 1.0, 0.5
taus = [1.0, 5.0, 20.0]                  # the pre-registered retention band (s)
labels = [rf"$\tau_{{\rm leak}}={tl:g}$ s" for tl in taus]

fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.axvspan(coincidence_at, coincidence_at + coincidence_dur, color=GOLD, alpha=0.30, lw=0)
ax.text(coincidence_at + coincidence_dur / 2, 1.06, "coincidence", color=INK,
        fontsize=8, ha="center")
for tl, col, lab in zip(taus, VIR, labels):
    g = TransientGate(V=1.5, tau_leak=tl, dt=0.05)          # fitted device @ V=1.5
    e = g.trace(t, coincidence_at=coincidence_at, coincidence_dur=coincidence_dur)
    ax.plot(t, e, color=col, lw=2.0, label=lab)
    ax.fill_between(t, 0, e, color=col, alpha=0.10, lw=0)
ax.set_xlabel("time after coincidence (s)"); ax.set_ylabel(r"eligibility trace $e(t)$ (norm.)")
ax.set_xlim(0, 80); ax.set_ylim(0, 1.12)
ax.set_title(r"Device trace: compressed-exponential rise, $\tau_{\rm leak}$-set decay")
ax.legend(frameon=False, fontsize=9, title="deep traps  <->  shallow traps",
          title_fontsize=8); _clean(ax); plt.show()

# half-peak decay delay grows with tau_leak -- the device-physics origin of the window
for tl in taus:
    g = TransientGate(V=1.5, tau_leak=tl, dt=0.05)
    e = g.trace(t, coincidence_at=coincidence_at, coincidence_dur=coincidence_dur)
    pk = t[int(np.argmax(e))]
    post_m = t > pk
    tp, ep = t[post_m], e[post_m]
    d_half = (tp[int(np.argmin(np.abs(ep - 0.5)))] - pk) if (ep < 0.5).any() else float("nan")
    print(f"  tau_leak={tl:5.1f} s: peak @ {pk:4.1f} s, half-peak decay ~ {d_half:5.1f} s")

In [ ]:
if QUICK:
    from mrl_trace.bandit import run_learning_and_window
    r = run_learning_and_window(seeds=6)
else:
    r = paths.load_result("tier3_results.npy")

delays = np.asarray(r["delays"], float)          # [1, 2, 5, 10, 20]
taus = sorted(r["reward_rate"].keys(), reverse=True)   # [10.0, 2.0, 0.5]
tcol = {10.0: VIR[0], 2.0: VIR[1], 0.5: VIR[2]}
W = 50                                            # running-mean window (matches Fig 6)

def _running(rw_2d, w=W):
    cs = np.cumsum(np.insert(rw_2d, 0, 0.0, axis=1), axis=1)
    rr = (cs[:, w:] - cs[:, :-w]) / w             # (seeds, trials-w)
    return rr.mean(0), np.percentile(rr, 2.5, axis=0), np.percentile(rr, 97.5, axis=0)

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 3.8))

# (a) delay x retention reward-rate table with 95% CI bands
for tl in taus:
    y = np.asarray(r["reward_rate"][tl], float)
    ci = r["reward_rate_ci"][tl]
    lo = np.array([c[0] for c in ci]); hi = np.array([c[1] for c in ci])
    axA.plot(delays, y, "-o", ms=4, lw=1.7, color=tcol[tl],
             label=rf"$\tau_{{\rm leak}}={tl:g}$ s")
    axA.fill_between(delays, lo, hi, color=tcol[tl], alpha=0.18, lw=0)
axA.axhline(r.get("crit", 0.75), ls="--", color=GREY, lw=1.0)
axA.axhline(0.5, ls=":", color=RED, lw=1.0)
axA.text(delays[-1], 0.505, "chance", fontsize=7.5, ha="right", va="bottom", color=RED)
axA.set_xscale("log"); axA.set_xticks(delays)
axA.set_xticklabels([f"{d:g}" for d in delays])
axA.set_xlabel(r"action$\to$reward delay $D$ (s)"); axA.set_ylabel("final reward rate")
axA.set_ylim(0.45, 1.03); axA.set_title("(a) delay x retention window", fontsize=10)
axA.legend(fontsize=8, frameon=False, loc="upper right"); _clean(axA)

# (b) device vs no-trace learning curve (running reward rate, 95% CI band)
for arr, col, lab in [(r["curve_device"], GREEN, "device trace"),
                      (r["curve_notrace"], GREY, "no-trace")]:
    m, lo, hi = _running(np.asarray(arr, float))
    x = np.arange(W, W + len(m))
    axB.plot(x, m, color=col, lw=1.8, label=lab, zorder=4)
    axB.fill_between(x, lo, hi, color=col, alpha=0.20, lw=0, zorder=2)
axB.axhline(r.get("crit", 0.75), ls="--", color=GREY, lw=1.0)
axB.axhline(0.5, ls=":", color=RED, lw=1.0, label="chance")
axB.set_xlabel("trial"); axB.set_ylabel("reward rate (running)")
axB.set_ylim(0.3, 1.03); axB.set_title(rf"(b) learning curve ($\tau_{{\rm leak}}=10$ s, $D=D_0={r['D0']:g}$ s)",
                                       fontsize=10)
axB.legend(fontsize=8, frameon=False, loc="center right"); _clean(axB)
plt.show()

dev, nt = r["device_final"], r["notrace_final"]
print(f"C2 device learns : final reward rate {dev[0]:.3f}  95% CI [{dev[2]:.3f}, {dev[3]:.3f}]  "
      f"({'PASS' if dev[0] >= r.get('crit', 0.75) else 'fail'} vs crit {r.get('crit', 0.75)})")
print(f"C1 necessity     : no-trace {nt[0]:.3f}  95% CI [{nt[2]:.3f}, {nt[3]:.3f}]  "
      f"({'PASS' if nt[0] <= 0.5 + 0.10 else 'fail'} vs chance+0.10)")
print("max learnable delay index per retention (C4 monotone in tau):",
      {tl: r['max_learn'][tl] for tl in taus})

In [ ]:
# Replay-only: the dense 13x18 x 20-seed grid is too heavy for an in-kernel default.
r = paths.load_result("exp8_dmax_law.npy")
taus = np.asarray(r["taus"], float)
dvals = np.array([r["dmax"][t] for t in r["taus"]], float)
censored = np.asarray(r["censored"], bool)
k, r2 = r["k"], r["r2"]
k_plateau, plateau_tau, crit = r["k_plateau"], r["plateau_tau"], r["crit"]

# fit / plateau masks (mirror the exp8 driver): resolved = not censored & D_max>0
fitmask = (~censored) & (dvals > 0)
rise = fitmask & (taus < plateau_tau)
plat = fitmask & (taus >= plateau_tau)

fig, ax = plt.subplots(figsize=(6.4, 4.4))
xx = np.linspace(0, taus.max() * 1.05, 100)
# headline origin fit over all resolved points
ax.plot(xx, k * xx, color=INK, lw=1.3, ls="--", zorder=2,
        label=rf"$D_{{\max}} = {k:.0f}\,\tau_{{\rm leak}}$  ($R^2={r2:.2f}$, all points)")
# asymptotic plateau slope (steeper; the trace-dominated regime)
ax.plot(xx, k_plateau * xx, color=RED, lw=1.1, ls=(0, (5, 2)), zorder=2,
        label=rf"plateau slope $\approx{k_plateau:.0f}$ ($\tau_{{\rm leak}}\geq{plateau_tau:g}$ s)")
ax.scatter(taus[rise], dvals[rise], s=58, color=INDIGO, edgecolor="white", zorder=4,
           label="rise-limited / transition")
ax.scatter(taus[plat], dvals[plat], s=58, marker="s", color=GREEN, edgecolor="white",
           zorder=4, label="trace-dominated plateau")
if censored.any():
    ax.scatter(taus[censored], dvals[censored], s=58, marker="^", color=GOLD,
               edgecolor="white", zorder=4, label=r"censored ($D_{\max}\geq$ grid max)")
ax.set_xlabel(r"retention $\tau_{\rm leak}$ (s)"); ax.set_ylabel(r"max learnable delay $D_{\max}$ (s)")
ax.set_title(rf"Densely-sampled retention-delay scaling (crit = {crit:g})", fontsize=10)
ax.legend(frameon=False, fontsize=8, loc="upper left"); _clean(ax); plt.show()

print(f"origin fit over {int(fitmask.sum())} resolved points: D_max = {k:.2f} * tau_leak, R^2 = {r2:.3f}")
print(f"trace-dominated plateau slope (tau_leak >= {plateau_tau:g} s): D_max/tau_leak -> {k_plateau:.2f}")
if censored.any():
    print("right-censored at grid ceiling (D_max >= max delay): tau_leak =",
          [float(t) for t in taus[censored]])
else:
    print("no right-censored points at this grid ceiling")
print("D_max per retention (s):", {float(t): round(float(r['dmax'][t]), 1) for t in r['taus']})

In [ ]:
# Uncomment to launch the full sweeps as subprocesses (streamed; the exp8 grid is heavy):
# import subprocess, sys
# subprocess.run([sys.executable, "-m", "mrl_trace.bandit", "--tier3", "--full"])
print("see the markdown above for the full-scale commands")